# PPT Asset Generator V2

This notebook generates slide-ready visual assets for the final Jeonse price prediction presentation.

V2 is designed for PPT use, not exploratory analysis:

- `figure_only/`: clean cropped tables and charts for figure placeholders
- `full_slide/`: 16:9 PNG slides that can replace a whole PPT slide if needed
- `asset_manifest.csv`: which asset should go into which slide
- `ppt_assets_v2.zip`: all generated assets

Run all cells from top to bottom in Colab.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/hyeon03-sketch/IML-Final-project.git"
BRANCH = "codex/ml-project-review"
REPO_DIR = Path("/content/IML-Final-project")

if Path("/content").exists():
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)
else:
    os.chdir(Path.cwd())

print("Working directory:", Path.cwd())
print("Dataset exists:", Path("IML_Final_dataset.xlsx").exists())

required = ["pandas", "numpy", "matplotlib", "seaborn", "sklearn", "xgboost", "shap", "openpyxl", "PIL"]
missing = []
for package in required:
    try:
        __import__(package)
    except Exception:
        missing.append(package)

if missing:
    install_names = []
    for package in missing:
        if package == "sklearn":
            install_names.append("scikit-learn")
        elif package == "PIL":
            install_names.append("pillow")
        else:
            install_names.append(package)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *install_names])
    print("Installed:", install_names)
else:
    print("All required packages are already installed.")


In [ ]:
import warnings
from pathlib import Path
import shutil
import textwrap
import subprocess

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import shap

from PIL import Image, ImageDraw, ImageFont
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupShuffleSplit, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

RND = 42
DATA_PATH = Path("IML_Final_dataset.xlsx")
SHEET_NAME = "최종데이터셋_모델용"
TARGET = "전세환산보증금(만원)"
CONVERSION_RATE = 0.065

OUT_DIR = Path("ppt_assets_v2")
FIG_DIR = OUT_DIR / "figure_only"
SLIDE_DIR = OUT_DIR / "full_slide"
CSV_DIR = OUT_DIR / "csv"
for d in [OUT_DIR, FIG_DIR, SLIDE_DIR, CSV_DIR]:
    d.mkdir(parents=True, exist_ok=True)

COLORS = {
    "blue": "#2F5FA8",
    "blue2": "#3268B1",
    "cyan": "#20AFC2",
    "orange": "#F27A21",
    "pale": "#EEF3FA",
    "light": "#F6F8FB",
    "dark": "#1F2A44",
    "gray": "#6B7280",
    "line": "#D6DEE9",
    "green": "#2E8B57",
    "red": "#C2410C",
    "white": "#FFFFFF",
}

plt.rcParams["figure.dpi"] = 130
pd.set_option("display.max_columns", 120)


def setup_fonts():
    candidate_paths = [
        Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf"),
        Path("/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf"),
        Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
        Path("/System/Library/Fonts/AppleSDGothicNeo.ttc"),
        Path("/Library/Fonts/Arial Unicode.ttf"),
    ]
    if not any(p.exists() for p in candidate_paths[:2]):
        try:
            subprocess.check_call(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            subprocess.check_call(["apt-get", "install", "-y", "fonts-nanum"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except Exception as exc:
            print("Font installation skipped:", exc)
    for cache_file in Path.home().glob(".cache/matplotlib/fontlist*"):
        try:
            cache_file.unlink()
        except Exception:
            pass
    usable = [p for p in candidate_paths if p.exists()]
    for font_path in usable:
        try:
            fm.fontManager.addfont(str(font_path))
        except Exception:
            pass
    try:
        fm._load_fontmanager(try_read_cache=False)
    except Exception:
        pass
    mpl_font = "DejaVu Sans"
    pil_path = usable[0] if usable else None
    if usable:
        try:
            mpl_font = fm.FontProperties(fname=str(usable[0])).get_name()
        except Exception:
            pass
    mpl.rcParams["font.family"] = [mpl_font]
    mpl.rcParams["font.sans-serif"] = [mpl_font]
    mpl.rcParams["axes.unicode_minus"] = False
    sns.set_theme(style="whitegrid", font=mpl_font)
    print("Matplotlib font:", mpl_font)
    print("PIL font:", pil_path)
    return pil_path


FONT_PATH = setup_fonts()


def font(size, bold=False):
    if FONT_PATH:
        return ImageFont.truetype(str(FONT_PATH), size=size)
    return ImageFont.load_default()


def hex_to_rgb(value):
    value = value.lstrip("#")
    return tuple(int(value[i:i+2], 16) for i in (0, 2, 4))


def text_size(draw, text, fnt):
    bbox = draw.textbbox((0, 0), text, font=fnt)
    return bbox[2] - bbox[0], bbox[3] - bbox[1]


def wrap_to_width(draw, text, fnt, max_width):
    words = str(text).split()
    if not words:
        return [""]
    lines, line = [], words[0]
    for word in words[1:]:
        candidate = line + " " + word
        if text_size(draw, candidate, fnt)[0] <= max_width:
            line = candidate
        else:
            lines.append(line)
            line = word
    lines.append(line)
    return lines


def draw_text(draw, xy, text, fnt, fill, max_width=None, line_gap=6, anchor="la", align="left"):
    x, y = xy
    if max_width is None:
        draw.text((x, y), str(text), font=fnt, fill=fill, anchor=anchor)
        return
    lines = []
    for part in str(text).split("\n"):
        lines.extend(wrap_to_width(draw, part, fnt, max_width))
    cur_y = y
    for line in lines:
        tw, th = text_size(draw, line, fnt)
        if align == "center":
            tx = x + (max_width - tw) / 2
        elif align == "right":
            tx = x + max_width - tw
        else:
            tx = x
        draw.text((tx, cur_y), line, font=fnt, fill=fill)
        cur_y += th + line_gap


def canvas(width=1600, height=900, bg="#FFFFFF"):
    return Image.new("RGB", (width, height), hex_to_rgb(bg))


def save_img(img, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    img.save(path, quality=95)
    print("Saved:", path)
    return path


def draw_panel(draw, box, fill="#FFFFFF", outline="#D6DEE9", radius=22, width=2):
    draw.rounded_rectangle(box, radius=radius, fill=hex_to_rgb(fill), outline=hex_to_rgb(outline), width=width)


def draw_header(draw, title, subtitle=None, section=None, width=1600):
    if section:
        draw_text(draw, (70, 46), section.upper(), font(22), hex_to_rgb(COLORS["orange"]))
    draw_text(draw, (70, 82), title.upper(), font(56), hex_to_rgb(COLORS["blue"]))
    if subtitle:
        draw_text(draw, (74, 158), subtitle, font(24), hex_to_rgb(COLORS["dark"]), max_width=1280)


def make_table_image(rows, col_widths, title=None, subtitle=None, out_path=None, width=None,
                     row_height=74, header_height=76, font_size=26, header_font_size=27):
    width = width or sum(col_widths) + 80
    top = 120 if title else 45
    height = top + header_height + row_height * (len(rows) - 1) + 50
    img = canvas(width, height)
    d = ImageDraw.Draw(img)
    if title:
        draw_text(d, (40, 34), title, font(38), hex_to_rgb(COLORS["blue"]))
    if subtitle:
        draw_text(d, (42, 82), subtitle, font(20), hex_to_rgb(COLORS["gray"]), max_width=width-90)
    x0, y0 = 40, top
    x = x0
    for j, cw in enumerate(col_widths):
        d.rectangle([x, y0, x+cw, y0+header_height], fill=hex_to_rgb(COLORS["blue"]))
        draw_text(d, (x+14, y0+22), rows[0][j], font(header_font_size), hex_to_rgb("#FFFFFF"), max_width=cw-28, align="center")
        x += cw
    for i, row in enumerate(rows[1:], start=1):
        y = y0 + header_height + (i-1) * row_height
        fill = COLORS["light"] if i % 2 == 0 else "#FFFFFF"
        x = x0
        for j, cw in enumerate(col_widths):
            d.rectangle([x, y, x+cw, y+row_height], fill=hex_to_rgb(fill), outline=hex_to_rgb(COLORS["line"]), width=1)
            draw_text(d, (x+16, y+18), row[j], font(font_size), hex_to_rgb(COLORS["dark"]), max_width=cw-32, align="center")
            x += cw
    if out_path:
        save_img(img, out_path)
    return img


def make_title_card(title, body, out_path, width=1600, height=900):
    img = canvas(width, height)
    d = ImageDraw.Draw(img)
    draw_header(d, title, body)
    save_img(img, out_path)
    return img


In [ ]:
def make_jeonse_equivalent_target(data, conversion_rate=CONVERSION_RATE):
    out = data.copy()
    deposit = pd.to_numeric(out["보증금(만원)"], errors="coerce")
    monthly_rent = pd.to_numeric(out["월세금(만원)"], errors="coerce").fillna(0)
    lease_type = out["전월세구분"].astype(str).str.strip()
    target = deposit.astype(float).copy()
    monthly_mask = lease_type.eq("월세")
    target.loc[monthly_mask] = deposit.loc[monthly_mask] + monthly_rent.loc[monthly_mask] * 12.0 / conversion_rate
    out[TARGET] = target
    return out


def first_mode(series):
    modes = series.mode(dropna=False)
    return modes.iloc[0] if len(modes) else np.nan


def harmonize_dong_level_features(data, feature_cols, group_col="읍면동"):
    out = data.copy()
    rows = []
    for col in feature_cols:
        unique_by_dong = out.groupby(group_col)[col].nunique(dropna=False)
        inconsistent = unique_by_dong[unique_by_dong > 1]
        before = out[col].copy()
        out[col] = out.groupby(group_col)[col].transform(first_mode)
        changed = int((before.astype(str) != out[col].astype(str)).sum())
        rows.append({
            "Feature": col,
            "Dongs inconsistent before": len(inconsistent),
            "Rows harmonized": changed,
            "Max unique values after": int(out.groupby(group_col)[col].nunique(dropna=False).max()),
        })
    return out, pd.DataFrame(rows)


DONG_LEVEL_COLS = [
    "바다여부", "공원여부", "공원수", "최대공원면적(㎡)", "총공원면적(㎡)",
    "동_초등학교수", "동_중학교수", "동_고등학교수", "동_총학교수", "동_초중고모두있음여부",
]

raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
df = make_jeonse_equivalent_target(raw)
df = df[df[TARGET].notna() & df[TARGET].gt(0)].copy()
df, harmonization_report = harmonize_dong_level_features(df, DONG_LEVEL_COLS)

dataset_summary = pd.DataFrame([
    ["Total observations", f"{len(df):,}"],
    ["Jeonse transactions", f"{int(df['전월세구분'].eq('전세').sum()):,}"],
    ["Monthly-rent transactions converted", f"{int(df['전월세구분'].eq('월세').sum()):,}"],
    ["Number of dongs", f"{df['읍면동'].nunique():,}"],
    ["Target mean", f"{df[TARGET].mean():,.2f}"],
    ["Target median", f"{df[TARGET].median():,.2f}"],
], columns=["Item", "Value"])

display(dataset_summary)
display(harmonization_report)

dataset_summary.to_csv(CSV_DIR / "dataset_summary.csv", index=False, encoding="utf-8-sig")
harmonization_report.to_csv(CSV_DIR / "dong_harmonization_report.csv", index=False, encoding="utf-8-sig")


In [ ]:
BASE_FEATURES = ["전용면적(㎡)", "층", "건물연령(계약기준)", "계약연도", "계약월"]
DONG_FEATURE = ["읍면동"]
SPATIAL_FEATURES = [
    "바다여부",
    "공원여부",
    "공원수",
    "최대공원면적(㎡)",
    "동_초등학교수",
    "동_중학교수",
    "동_고등학교수",
    "동_초중고모두있음여부",
]
FEATURE_GROUPS = {
    "A_location_baseline": BASE_FEATURES + DONG_FEATURE,
    "B_location_spatial": BASE_FEATURES + DONG_FEATURE + SPATIAL_FEATURES,
}
BINARY_COLS = {"바다여부", "공원여부", "동_초중고모두있음여부"}
CATEGORICAL_COLS = {"읍면동"}


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def split_feature_types(cols):
    categorical = [c for c in cols if c in CATEGORICAL_COLS]
    binary = [c for c in cols if c in BINARY_COLS]
    numeric = [c for c in cols if c not in set(categorical + binary)]
    return numeric, categorical, binary


def make_preprocessor(cols):
    numeric, categorical, binary = split_feature_types(cols)
    transformers = []
    pass_cols = numeric + binary
    if pass_cols:
        transformers.append(("pass", "passthrough", pass_cols))
    if categorical:
        transformers.append(("cat", make_one_hot_encoder(), categorical))
    return ColumnTransformer(transformers=transformers, remainder="drop")


def make_xgboost():
    return XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        random_state=RND,
        n_jobs=-1,
        importance_type="gain",
    )


XGB_PARAM_GRID = {
    "model__n_estimators": [300, 600],
    "model__max_depth": [4, 6],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.9],
    "model__colsample_bytree": [0.9],
}


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape_pct(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true > 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def evaluate_predictions(y_true, pred):
    return {
        "test_rmse": rmse(y_true, pred),
        "test_mae": float(mean_absolute_error(y_true, pred)),
        "test_mape_pct": mape_pct(y_true, pred),
        "test_r2": float(r2_score(y_true, pred)),
    }


def fit_xgb_grid(data, group_name, cols, train_idx, test_idx):
    X = data[cols].copy()
    y = data[TARGET].copy()
    X_train, X_test = X.loc[train_idx], X.loc[test_idx]
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]

    pipe = Pipeline([("pre", make_preprocessor(cols)), ("model", make_xgboost())])
    search = GridSearchCV(
        pipe,
        param_grid=XGB_PARAM_GRID,
        scoring="neg_root_mean_squared_error",
        cv=KFold(n_splits=5, shuffle=True, random_state=RND),
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    pred = search.predict(X_test)
    metrics = evaluate_predictions(y_test, pred)
    metrics.update({
        "experiment": group_name,
        "model": "XGBoost",
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "cv_rmse": float(-search.best_score_),
        "best_params": search.best_params_,
    })
    fitted = {"estimator": search.best_estimator_, "X_test": X_test, "y_test": y_test, "pred": pred}
    return metrics, fitted


train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=RND)
rows, fitted = [], {}
for group_name, cols in FEATURE_GROUPS.items():
    print("Running", group_name)
    metrics, bundle = fit_xgb_grid(df, group_name, cols, train_idx, test_idx)
    rows.append(metrics)
    fitted[group_name] = bundle
    print(f"  RMSE={metrics['test_rmse']:.2f}, MAE={metrics['test_mae']:.2f}, MAPE={metrics['test_mape_pct']:.2f}%, R2={metrics['test_r2']:.4f}")

results = pd.DataFrame(rows).sort_values("test_rmse")
a = results[results["experiment"].eq("A_location_baseline")].iloc[0]
b = results[results["experiment"].eq("B_location_spatial")].iloc[0]
main_delta = pd.DataFrame([{
    "Comparison": "B_location_spatial - A_location_baseline",
    "RMSE change": b["test_rmse"] - a["test_rmse"],
    "MAE change": b["test_mae"] - a["test_mae"],
    "MAPE change (%p)": b["test_mape_pct"] - a["test_mape_pct"],
    "R² change": b["test_r2"] - a["test_r2"],
}])

display(results[["experiment", "test_rmse", "test_mae", "test_mape_pct", "test_r2"]])
display(main_delta)
results.to_csv(CSV_DIR / "main_results.csv", index=False, encoding="utf-8-sig")
main_delta.to_csv(CSV_DIR / "main_delta.csv", index=False, encoding="utf-8-sig")


In [ ]:
def run_holdout(validation_name, train_idx, test_idx):
    rows = []
    for group_name, cols in FEATURE_GROUPS.items():
        print("Strict", validation_name, group_name)
        metrics, _ = fit_xgb_grid(df, group_name, cols, train_idx, test_idx)
        metrics["validation"] = validation_name
        rows.append(metrics)
    return rows


strict_rows = []
train_idx_s, test_idx_s = train_test_split(df.index, test_size=0.2, random_state=RND)
strict_rows.extend(run_holdout("random_split", train_idx_s, test_idx_s))

years = sorted(df["계약연도"].dropna().unique())
if len(years) >= 2:
    latest_year = years[-1]
    strict_rows.extend(run_holdout(
        f"time_holdout_test_{latest_year}",
        df.index[df["계약연도"] < latest_year],
        df.index[df["계약연도"] == latest_year],
    ))

complex_groups = df["단지명"].fillna("missing_complex")
train_pos, test_pos = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RND).split(df, groups=complex_groups))
strict_rows.extend(run_holdout("complex_holdout", df.index[train_pos], df.index[test_pos]))

dong_groups = df["읍면동"].fillna("missing_dong")
train_pos, test_pos = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RND).split(df, groups=dong_groups))
strict_rows.extend(run_holdout("dong_holdout", df.index[train_pos], df.index[test_pos]))

strict_results = pd.DataFrame(strict_rows).sort_values(["validation", "experiment"])
delta_rows = []
for validation in strict_results["validation"].unique():
    sub = strict_results[strict_results["validation"].eq(validation)]
    a = sub[sub["experiment"].eq("A_location_baseline")].iloc[0]
    b = sub[sub["experiment"].eq("B_location_spatial")].iloc[0]
    delta_rows.append({
        "Validation": validation.replace("_", " "),
        "RMSE change": b["test_rmse"] - a["test_rmse"],
        "MAE change": b["test_mae"] - a["test_mae"],
        "MAPE change (%p)": b["test_mape_pct"] - a["test_mape_pct"],
        "R² change": b["test_r2"] - a["test_r2"],
    })
strict_delta = pd.DataFrame(delta_rows)

display(strict_results[["validation", "experiment", "test_rmse", "test_mae", "test_mape_pct", "test_r2"]])
display(strict_delta)
strict_results.to_csv(CSV_DIR / "strict_validation_results.csv", index=False, encoding="utf-8-sig")
strict_delta.to_csv(CSV_DIR / "strict_validation_delta.csv", index=False, encoding="utf-8-sig")


In [ ]:
def classify_feature_group(feature_name):
    clean = feature_name.replace("pass__", "").replace("cat__", "")
    if clean.startswith("읍면동_"):
        return "Dong location"
    if any(x in clean for x in ["전용면적", "층", "건물연령"]):
        return "Housing structure"
    if any(x in clean for x in ["계약연도", "계약월"]):
        return "Contract time"
    if any(x in clean for x in ["초등학교", "중학교", "고등학교", "초중고"]):
        return "School"
    if any(x in clean for x in ["공원"]):
        return "Park"
    if any(x in clean for x in ["바다"]):
        return "Sea"
    return "Other"


best_group = "B_location_spatial"
pipe = fitted[best_group]["estimator"]
pre = pipe.named_steps["pre"]
model = pipe.named_steps["model"]

X_for_shap = df[FEATURE_GROUPS[best_group]].copy()
X_sample = X_for_shap.sample(min(800, len(X_for_shap)), random_state=RND)
X_transformed = pre.transform(X_sample)
feature_names = pre.get_feature_names_out()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_transformed)
mean_abs = np.abs(shap_values).mean(axis=0)

shap_importance = pd.DataFrame({"Feature": feature_names, "Mean |SHAP|": mean_abs})
shap_importance["Feature group"] = shap_importance["Feature"].map(classify_feature_group)
shap_importance = shap_importance.sort_values("Mean |SHAP|", ascending=False)
group_shap = shap_importance.groupby("Feature group", as_index=False)["Mean |SHAP|"].sum().sort_values("Mean |SHAP|", ascending=False)
group_shap["SHAP share (%)"] = group_shap["Mean |SHAP|"] / group_shap["Mean |SHAP|"].sum() * 100

display(group_shap)
display(shap_importance.head(20))
group_shap.to_csv(CSV_DIR / "shap_group_importance.csv", index=False, encoding="utf-8-sig")
shap_importance.to_csv(CSV_DIR / "shap_feature_importance.csv", index=False, encoding="utf-8-sig")


In [ ]:
# Figure-only assets: compact, aligned, and ready for PPT placeholders.

make_table_image(
    [["Item", "Value"], *dataset_summary.values.tolist()],
    [520, 360],
    title="Final Modeling Dataset",
    subtitle="Buk-gu, Pohang apartment lease transactions",
    out_path=FIG_DIR / "fig_13_dataset_overview_table.png",
    font_size=24,
)

make_table_image(
    [
        ["Transaction type", "Target construction"],
        ["Jeonse", "Jeonse-equivalent deposit = deposit"],
        ["Monthly rent", "Jeonse-equivalent deposit = deposit + monthly rent x 12 / 0.065"],
    ],
    [360, 820],
    title="Target Construction",
    subtitle="The model predicts jeonse-equivalent deposit, not raw deposit.",
    out_path=FIG_DIR / "fig_14_target_formula_table.png",
    font_size=22,
    row_height=88,
)

spatial_rows = [
    ["Category", "Variables", "Interpretation"],
    ["Sea", "sea proximity dummy", "Coastal or near-coastal dong"],
    ["Park", "park existence, park count, max park area", "Green-space availability and scale"],
    ["School", "elementary, middle, high school counts", "Educational infrastructure"],
    ["Education mix", "all three school levels present", "Full school-level mix"],
]
make_table_image(
    spatial_rows,
    [260, 560, 500],
    title="Spatial-Derived Variables",
    subtitle="Area-level variables used to represent local neighborhood context.",
    out_path=FIG_DIR / "fig_15_spatial_variables_table.png",
    font_size=21,
)

feature_rows = [
    ["Experiment", "Purpose", "Included variables"],
    ["A_location_baseline", "Location-controlled baseline", "area, floor, building age, contract year/month, dong"],
    ["B_location_spatial", "Spatial-extended model", "A + sea, park, and school-related variables"],
]
make_table_image(
    feature_rows,
    [360, 420, 760],
    title="Feature Set Design",
    subtitle="Dong is included in the baseline as a location control.",
    out_path=FIG_DIR / "fig_17_feature_set_table.png",
    font_size=21,
    row_height=92,
)

harm_small = harmonization_report.copy()
harm_small = harm_small.rename(columns={
    "Dongs inconsistent before": "Inconsistent dongs",
    "Rows harmonized": "Rows fixed",
    "Max unique values after": "Max unique after",
})
harm_small["Feature"] = harm_small["Feature"].replace({
    "동_초등학교수": "Elementary school count",
    "동_총학교수": "Total school count",
    "동_중학교수": "Middle school count",
    "동_고등학교수": "High school count",
    "동_초중고모두있음여부": "All school levels present",
    "바다여부": "Sea dummy",
    "공원여부": "Park dummy",
    "공원수": "Park count",
    "최대공원면적(㎡)": "Max park area",
    "총공원면적(㎡)": "Total park area",
})
make_table_image(
    [harm_small.columns.tolist(), *harm_small.values.tolist()],
    [470, 260, 210, 240],
    title="Dong-Level Consistency Check",
    subtitle="Dong-level variables were harmonized using the within-dong mode.",
    out_path=FIG_DIR / "fig_16_dong_harmonization_table.png",
    font_size=19,
    row_height=62,
    header_height=72,
)


In [ ]:
# Clean charts for PPT placeholders.

def save_matplotlib(path):
    plt.savefig(path, dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Saved:", path)


lease_counts = df["전월세구분"].value_counts().reindex(["전세", "월세"]).fillna(0)
fig, ax = plt.subplots(figsize=(8.5, 4.8))
bars = ax.bar(["Jeonse", "Monthly rent\nconverted"], lease_counts.values, color=[COLORS["blue"], COLORS["cyan"]], width=0.52)
ax.set_title("Transaction Composition", fontsize=18, weight="bold", color=COLORS["blue"], pad=14)
ax.set_ylabel("Number of observations")
ax.grid(axis="y", alpha=0.22)
ax.spines[["top", "right"]].set_visible(False)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+60, f"{int(b.get_height()):,}", ha="center", va="bottom", fontsize=12, weight="bold")
save_matplotlib(FIG_DIR / "fig_13_transaction_composition.png")

fig, ax = plt.subplots(figsize=(9, 4.8))
sns.histplot(df[TARGET], bins=42, kde=True, color=COLORS["blue"], ax=ax)
ax.axvline(df[TARGET].mean(), color=COLORS["orange"], linestyle="--", linewidth=2, label=f"Mean: {df[TARGET].mean():,.0f}")
ax.axvline(df[TARGET].median(), color=COLORS["green"], linestyle="-", linewidth=2, label=f"Median: {df[TARGET].median():,.0f}")
ax.set_title("Distribution of Jeonse-Equivalent Deposit", fontsize=18, weight="bold", color=COLORS["blue"], pad=14)
ax.set_xlabel("Jeonse-equivalent deposit (10,000 KRW)")
ax.set_ylabel("Count")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
save_matplotlib(FIG_DIR / "fig_14_target_distribution.png")

dong_summary = (
    df.groupby("읍면동")
    .agg(
        transactions=(TARGET, "size"),
        mean_target=(TARGET, "mean"),
        median_target=(TARGET, "median"),
        mean_area=("전용면적(㎡)", "mean"),
        mean_age=("건물연령(계약기준)", "mean"),
        sea=("바다여부", "first"),
        parks=("공원수", "first"),
        elem=("동_초등학교수", "first"),
        middle=("동_중학교수", "first"),
        high=("동_고등학교수", "first"),
    )
    .reset_index()
    .sort_values("mean_target", ascending=False)
)
dong_summary.to_csv(CSV_DIR / "dong_summary.csv", index=False, encoding="utf-8-sig")

top_dong = dong_summary.head(14).sort_values("mean_target")
fig, ax = plt.subplots(figsize=(9, 6.2))
ax.barh(top_dong["읍면동"], top_dong["mean_target"], color=COLORS["blue"])
ax.set_title("Average Jeonse-Equivalent Deposit by Dong", fontsize=18, weight="bold", color=COLORS["blue"], pad=14)
ax.set_xlabel("Average target (10,000 KRW)")
ax.grid(axis="x", alpha=0.22)
ax.spines[["top", "right"]].set_visible(False)
for i, v in enumerate(top_dong["mean_target"]):
    ax.text(v + 280, i, f"{v:,.0f}", va="center", fontsize=9)
save_matplotlib(FIG_DIR / "fig_18_dong_mean_target_bar.png")

result_plot = results.copy().sort_values("experiment")
fig, axes = plt.subplots(1, 4, figsize=(12.5, 3.7))
metrics = [("test_rmse", "RMSE"), ("test_mae", "MAE"), ("test_mape_pct", "MAPE (%)"), ("test_r2", "R²")]
for ax, (col, title) in zip(axes, metrics):
    vals = result_plot[col].values
    ax.bar(["A", "B"], vals, color=[COLORS["blue"], COLORS["cyan"]], width=0.55)
    ax.set_title(title, fontsize=13, weight="bold", color=COLORS["blue"])
    ax.grid(axis="y", alpha=0.18)
    ax.spines[["top", "right"]].set_visible(False)
    for i, v in enumerate(vals):
        label = f"{v:.4f}" if col == "test_r2" else f"{v:,.1f}"
        ax.text(i, v, label, ha="center", va="bottom", fontsize=8.5)
fig.suptitle("Main Result: Baseline vs Spatial Model", fontsize=18, weight="bold", color=COLORS["blue"])
plt.tight_layout()
save_matplotlib(FIG_DIR / "fig_21_main_metrics_comparison.png")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 5.0))
for ax, group_name in zip(axes, ["A_location_baseline", "B_location_spatial"]):
    bundle = fitted[group_name]
    ax.scatter(bundle["y_test"], bundle["pred"], s=10, alpha=0.42, color=COLORS["blue"])
    lim = max(bundle["y_test"].max(), np.max(bundle["pred"]))
    ax.plot([0, lim], [0, lim], "--", color=COLORS["orange"], linewidth=1.4)
    ax.set_title(group_name, fontsize=12, weight="bold", color=COLORS["blue"])
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.grid(alpha=0.18)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Prediction vs Actual", fontsize=18, weight="bold", color=COLORS["blue"])
plt.tight_layout()
save_matplotlib(FIG_DIR / "fig_25_prediction_vs_actual.png")


In [ ]:
# Holdout and SHAP visuals.

holdout_rows = [["Validation", "RMSE change", "MAPE change", "R² change"]]
for _, r in strict_delta.iterrows():
    holdout_rows.append([
        r["Validation"].replace("time holdout test ", "time holdout "),
        f"{r['RMSE change']:,.2f}",
        f"{r['MAPE change (%p)']:.3f}%p",
        f"{r['R² change']:.4f}",
    ])
make_table_image(
    holdout_rows,
    [460, 300, 300, 260],
    title="Holdout Validation Results",
    subtitle="Negative RMSE and MAPE changes indicate improvement.",
    out_path=FIG_DIR / "fig_24_holdout_delta_table.png",
    font_size=22,
)

fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.1))
for ax, col, title in zip(axes, ["RMSE change", "MAPE change (%p)", "R² change"], ["RMSE Change", "MAPE Change", "R² Change"]):
    vals = strict_delta[col].values
    colors = [COLORS["green"] if (v < 0 and col != "R² change") or (v > 0 and col == "R² change") else COLORS["red"] for v in vals]
    labels = strict_delta["Validation"].str.replace("time holdout test ", "time\n").str.replace("complex holdout", "complex\nholdout").str.replace("dong holdout", "dong\nholdout").str.replace("random split", "random\nsplit")
    ax.bar(labels, vals, color=colors, width=0.58)
    ax.axhline(0, color=COLORS["dark"], linewidth=0.9)
    ax.set_title(title, fontsize=13, weight="bold", color=COLORS["blue"])
    ax.grid(axis="y", alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)
    for i, v in enumerate(vals):
        label = f"{v:.3f}" if abs(v) < 10 else f"{v:.1f}"
        ax.text(i, v, label, ha="center", va="bottom" if v >= 0 else "top", fontsize=8.2)
fig.suptitle("Spatial Model Improvement Across Holdout Tests", fontsize=18, weight="bold", color=COLORS["blue"])
plt.tight_layout()
save_matplotlib(FIG_DIR / "fig_24_holdout_delta_bars.png")

group_rows = [["Feature group", "SHAP share"]]
for _, r in group_shap.iterrows():
    group_rows.append([r["Feature group"], f"{r['SHAP share (%)']:.2f}%"])
make_table_image(
    group_rows,
    [470, 290],
    title="SHAP Feature Group Contribution",
    subtitle="Housing structure is the dominant predictor.",
    out_path=FIG_DIR / "fig_27_shap_group_table.png",
    font_size=23,
)

fig, ax = plt.subplots(figsize=(8.6, 4.8))
plot_g = group_shap.sort_values("SHAP share (%)")
ax.barh(plot_g["Feature group"], plot_g["SHAP share (%)"], color=COLORS["blue"])
ax.set_title("SHAP Share by Feature Group", fontsize=18, weight="bold", color=COLORS["blue"], pad=14)
ax.set_xlabel("Share of total mean |SHAP| (%)")
ax.grid(axis="x", alpha=0.22)
ax.spines[["top", "right"]].set_visible(False)
for i, v in enumerate(plot_g["SHAP share (%)"]):
    ax.text(v + 0.6, i, f"{v:.2f}%", va="center", fontsize=10, weight="bold")
save_matplotlib(FIG_DIR / "fig_27_shap_group_bar.png")

top_features = shap_importance.head(16).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6.2))
ax.barh(top_features["Feature"].str.replace("pass__", "").str.replace("cat__", ""), top_features["Mean |SHAP|"], color=COLORS["green"])
ax.set_title("Top SHAP Features", fontsize=18, weight="bold", color=COLORS["blue"], pad=14)
ax.set_xlabel("Mean |SHAP|")
ax.grid(axis="x", alpha=0.22)
ax.spines[["top", "right"]].set_visible(False)
save_matplotlib(FIG_DIR / "fig_28_shap_top_features_bar.png")

plt.figure(figsize=(10, 5.8))
shap.summary_plot(shap_values, X_transformed, feature_names=feature_names, max_display=16, show=False)
plt.title("SHAP Summary Plot: B_location_spatial", fontsize=16, weight="bold", color=COLORS["blue"], pad=12)
save_matplotlib(FIG_DIR / "fig_28_shap_summary_plot.png")


In [ ]:
# Full-slide 16:9 assets. These can replace full PPT slides if desired.

def metric_card(draw, x, y, w, h, number, label, fill):
    draw_panel(draw, [x, y, x+w, y+h], fill=fill, outline=fill, radius=12)
    draw_text(draw, (x+20, y+22), number, font(40), hex_to_rgb("#FFFFFF"), max_width=w-40, align="center")
    draw_text(draw, (x+20, y+78), label, font(19), hex_to_rgb("#FFFFFF"), max_width=w-40, align="center")


def full_slide_dataset():
    img = canvas()
    d = ImageDraw.Draw(img)
    draw_header(d, "Final Dataset", "The final dataset combines jeonse rows and converted monthly-rent rows.", "Data & Method")
    cards = [("4,470", "Total observations"), ("2,486", "Jeonse rows"), ("1,984", "Converted monthly-rent rows"), ("26", "Dongs")]
    for i, (num, label) in enumerate(cards):
        metric_card(d, 70 + i*360, 250, 300, 120, num, label, COLORS["blue"] if i % 2 == 0 else COLORS["cyan"])
    tbl = make_table_image([["Item", "Value"], *dataset_summary.values.tolist()], [520, 360], width=960, font_size=22)
    img.paste(tbl.resize((760, 390)), (90, 430))
    draw_panel(d, [930, 430, 1450, 740], fill="#FFFFFF", outline=COLORS["blue"], radius=22, width=3)
    draw_text(d, (970, 470), "Figure space", font(30), hex_to_rgb(COLORS["blue"]))
    draw_text(d, (970, 540), "Insert figure_only/fig_13_transaction_composition.png", font(22), hex_to_rgb(COLORS["gray"]), max_width=430, align="center")
    save_img(img, SLIDE_DIR / "slide_13_final_dataset.png")


def full_slide_main_results():
    img = canvas()
    d = ImageDraw.Draw(img)
    draw_header(d, "Main XGBoost Result", "B_location_spatial improves every metric, but the random-split gain is small.", "Model Results")
    rows = [
        ["Experiment", "RMSE", "MAE", "MAPE", "R²"],
        ["A_location_baseline", "2,216.95", "1,518.12", "11.40%", "0.8969"],
        ["B_location_spatial", "2,208.91", "1,515.95", "11.39%", "0.8977"],
    ]
    tbl = make_table_image(rows, [420, 210, 210, 190, 150], width=1260, font_size=21, row_height=82)
    img.paste(tbl.resize((1050, 245)), (80, 230))
    rows2 = [["Change", "Value"], ["RMSE", "-8.04"], ["MAE", "-2.17"], ["MAPE", "-0.014%p"], ["R²", "+0.0007"]]
    tbl2 = make_table_image(rows2, [230, 210], width=520, font_size=20, row_height=58)
    img.paste(tbl2.resize((330, 245)), (1180, 230))
    draw_panel(d, [100, 560, 720, 800], fill="#FFFFFF", outline=COLORS["blue"], radius=18, width=3)
    draw_panel(d, [820, 560, 1440, 800], fill="#FFFFFF", outline=COLORS["blue"], radius=18, width=3)
    draw_text(d, (145, 605), "Figure space", font(26), hex_to_rgb(COLORS["blue"]))
    draw_text(d, (865, 605), "Figure space", font(26), hex_to_rgb(COLORS["blue"]))
    draw_text(d, (145, 665), "Insert fig_21_main_metrics_comparison.png", font(20), hex_to_rgb(COLORS["gray"]), max_width=530, align="center")
    draw_text(d, (865, 665), "Insert fig_25_prediction_vs_actual.png", font(20), hex_to_rgb(COLORS["gray"]), max_width=530, align="center")
    save_img(img, SLIDE_DIR / "slide_21_main_results.png")


def full_slide_holdout():
    img = canvas()
    d = ImageDraw.Draw(img)
    draw_header(d, "Holdout Validation Results", "The spatial model improved all stricter validation settings.", "Model Evaluation")
    tbl = Image.open(FIG_DIR / "fig_24_holdout_delta_table.png").resize((680, 310))
    img.paste(tbl, (80, 240))
    draw_panel(d, [820, 240, 1460, 550], fill="#FFFFFF", outline=COLORS["blue"], radius=18, width=3)
    draw_text(d, (860, 290), "Figure space", font(28), hex_to_rgb(COLORS["blue"]))
    draw_text(d, (860, 365), "Insert fig_24_holdout_delta_bars.png", font(22), hex_to_rgb(COLORS["gray"]), max_width=560, align="center")
    draw_panel(d, [130, 690, 1450, 785], fill=COLORS["blue"], outline=COLORS["blue"], radius=16, width=2)
    draw_text(d, (170, 722), "Main evidence", font(25), hex_to_rgb("#FFFFFF"))
    draw_text(d, (410, 722), "Spatial variables are more useful for generalization than for producing a large random-split gain.", font(22), hex_to_rgb("#FFFFFF"), max_width=950)
    save_img(img, SLIDE_DIR / "slide_24_holdout_results.png")


def full_slide_shap():
    img = canvas()
    d = ImageDraw.Draw(img)
    draw_header(d, "SHAP Feature Group Results", "Housing structure is the dominant source of prediction.", "Interpretation of Results")
    tbl = Image.open(FIG_DIR / "fig_27_shap_group_table.png").resize((600, 360))
    img.paste(tbl, (80, 220))
    draw_panel(d, [770, 230, 1450, 580], fill="#FFFFFF", outline=COLORS["blue"], radius=18, width=3)
    draw_text(d, (815, 280), "Figure space", font(28), hex_to_rgb(COLORS["blue"]))
    draw_text(d, (815, 355), "Insert fig_27_shap_group_bar.png", font(22), hex_to_rgb(COLORS["gray"]), max_width=590, align="center")
    draw_panel(d, [130, 720, 1450, 805], fill=COLORS["pale"], outline=COLORS["line"], radius=14, width=2)
    draw_text(d, (170, 748), "Interpretation", font(23), hex_to_rgb(COLORS["blue"]))
    draw_text(d, (410, 748), "Spatial variables are not the main predictors, but school and park variables provide additional local-context information.", font(21), hex_to_rgb(COLORS["dark"]), max_width=940)
    save_img(img, SLIDE_DIR / "slide_27_shap_results.png")


full_slide_dataset()
full_slide_main_results()
full_slide_holdout()
full_slide_shap()


In [ ]:
manifest_rows = [
    [13, "Dataset overview", "figure_only/fig_13_dataset_overview_table.png", "Use inside table placeholder"],
    [13, "Transaction composition", "figure_only/fig_13_transaction_composition.png", "Use as chart"],
    [14, "Target formula", "figure_only/fig_14_target_formula_table.png", "Use inside table placeholder"],
    [14, "Target distribution", "figure_only/fig_14_target_distribution.png", "Use as chart"],
    [15, "Spatial variables", "figure_only/fig_15_spatial_variables_table.png", "Use as table"],
    [16, "Dong harmonization", "figure_only/fig_16_dong_harmonization_table.png", "Use as table"],
    [17, "Feature set design", "figure_only/fig_17_feature_set_table.png", "Use as table"],
    [18, "Dong average target", "figure_only/fig_18_dong_mean_target_bar.png", "Use as chart"],
    [21, "Main metrics", "figure_only/fig_21_main_metrics_comparison.png", "Use as chart"],
    [25, "Prediction vs actual", "figure_only/fig_25_prediction_vs_actual.png", "Use as chart"],
    [24, "Holdout table", "figure_only/fig_24_holdout_delta_table.png", "Use as table"],
    [24, "Holdout bars", "figure_only/fig_24_holdout_delta_bars.png", "Use as chart"],
    [27, "SHAP group table", "figure_only/fig_27_shap_group_table.png", "Use as table"],
    [27, "SHAP group bar", "figure_only/fig_27_shap_group_bar.png", "Use as chart"],
    [28, "SHAP top features", "figure_only/fig_28_shap_top_features_bar.png", "Use as chart"],
    [28, "SHAP summary", "figure_only/fig_28_shap_summary_plot.png", "Use as chart"],
    [13, "Full slide dataset", "full_slide/slide_13_final_dataset.png", "Optional full-slide replacement"],
    [21, "Full slide main result", "full_slide/slide_21_main_results.png", "Optional full-slide replacement"],
    [24, "Full slide holdout", "full_slide/slide_24_holdout_results.png", "Optional full-slide replacement"],
    [27, "Full slide SHAP", "full_slide/slide_27_shap_results.png", "Optional full-slide replacement"],
]
asset_manifest = pd.DataFrame(manifest_rows, columns=["Slide", "Asset", "File", "Use"])
asset_manifest.to_csv(OUT_DIR / "asset_manifest.csv", index=False, encoding="utf-8-sig")
display(asset_manifest)

(OUT_DIR / "README.md").write_text(
    "# PPT Assets V2\n\n"
    "Use `figure_only/` for placeholders and `full_slide/` for optional full-slide replacement PNGs.\n\n"
    + "\n".join(f"- Slide {r.Slide}: `{r.File}` - {r.Use}" for _, r in asset_manifest.iterrows()),
    encoding="utf-8",
)

zip_path = shutil.make_archive("ppt_assets_v2", "zip", OUT_DIR)
print("Created:", zip_path)
print("Assets folder:", OUT_DIR.resolve())


In [ ]:
# Optional download in Colab:
# from google.colab import files
# files.download("ppt_assets_v2.zip")
